In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_19.pickle

In [ ]:
%%RecordEvent
%%time
### cell 20 ###

### cell 20 optimized ###

def bar_chart_multiple_years_29(df, x_column, y_column, title, y_axis_title):
    df.to_csv(
        '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results'
        '/kaggle/working/individual_charts/data/' + title + '.csv',
        index=True
    )

# Parameters (unchanged)
question_of_interest = 'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
column_of_interest = 'percentage'
title_for_chart   = 'Most common degree types on Kaggle from 2017-2022'
title_for_x_axis  = ''
title_for_y_axis  = '% of respondents'
subset_of_degrees = ["Bachelor's degree", "Master's degree", 'Doctoral degree']

# Whether to include 2017
include_2017_flag = True
year_list = ['2017','2018','2019','2020','2021','2022'] if include_2017_flag else ['2018','2019','2020','2021','2022']

# Build a single DataFrame of (year, degree)
dfs = []
for yr in year_list:
    df_yr = globals()[f"responses_df_{yr}"]
    dfs.append(
        df_yr[[question_of_interest]]
             .rename(columns={question_of_interest: 'degree'})
             .assign(year=yr)
    )
combined = pd.concat(dfs)

# Normalize inconsistent labels
mapping = {
    'Masterâ\x80\x99s degree': "Master's degree",
    'Bachelorâ\x80\x99s degree': "Bachelor's degree"
}
combined['degree'] = combined['degree'].replace(mapping)

# Compute counts and percentages in one GPU groupby
grouped = (
    combined
      .groupby(['year','degree'])
      .size()
      .reset_index(name='count')
)
grouped['percentage'] = (
    grouped['count'] * 100
    / grouped.groupby('year')['count'].transform('sum')
).round(1)

# Prepare final DataFrame
grouped = grouped.rename(columns={'degree': title_for_x_axis})
df_final = grouped[['percentage','year', title_for_x_axis]]
# Bucket all non-top degrees into 'Other'
df_final[title_for_x_axis] = (
    df_final[title_for_x_axis]
            .where(df_final[title_for_x_axis].isin(subset_of_degrees), 'Other')
)

# Persist CSV for downstream chartingar_chart_multiple_years_29(
    df_final,
    title_for_x_axis,
    column_of_interest,
    title_for_chart,
    title_for_y_axis
)


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_20_try_9.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_20_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[20], f)


In [ ]:
opt_output = Out.get(4)